In [55]:
import pandas as pd
from collections import defaultdict

d = pd.read_table('./df_new_all_lemma.tsv', sep='\t', on_bad_lines='skip')

df = d[['Smell_Word', 'year']].copy()
df['year'] = pd.to_numeric(df['year'], errors='coerce')

df = df.dropna(subset=['Smell_Word', 'year'])

word_freq_per_year = defaultdict(lambda: defaultdict(int))

for _, row in df.iterrows():
    year = int(row['year'])
    words = str(row['Smell_Word']).lower().split('|')  # Split sulle pipe
    for word in words:
        word = word.strip()
        if year and word:
            word_freq_per_year[year][word] += 1

records = []
for year, freqs in word_freq_per_year.items():
    for word, freq in freqs.items():
        records.append({'word': word, 'frequency': freq, 'year': year})

df_word_freq_year = pd.DataFrame(records)

df_word_freq_year = df_word_freq_year.sort_values(['year', 'frequency'], ascending=[True, False])

# df_word_freq_year


/var/folders/j8/2fw1pn3n4zb1y5l8gt8s56dc0000gn/T/ipykernel_55342/2361456401.py:4: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  d = pd.read_table('./df_new_all_lemma.tsv', sep='\t', on_bad_lines='skip')


In [33]:
word_freq_per_year = defaultdict(lambda: defaultdict(int))
for _, row in df.iterrows():
    year = int(row['year'])
    words = str(row['Smell_Word']).lower().split('|')
    for word in words:
        word = word.strip()
        if year and word:
            word_freq_per_year[year][word] += 1

records = []
for year, freqs in word_freq_per_year.items():
    for word, freq in freqs.items():
        records.append({'word': word, 'frequency': freq, 'year': year})

df_word_freq_year = pd.DataFrame(records).sort_values(['year', 'frequency'], ascending=[True, False])

In [30]:
df_word_freq_year = df_word_freq_year[df_word_freq_year['year'] < 2000]
df_word_freq_year = df_word_freq_year[df_word_freq_year['year'] > 1600]
df_word_freq_year

,word,frequency,year
66264,smell,75,1601
66270,odoriferous,29,1601
66267,stink,27,1601
66268,perfume,17,1601
66274,odour,15,1601
...,...,...,...
72290,toothpaste,1,1999
72295,perfumes,1,1999
72296,essential,1,1999
72297,aromas,1,1999


In [51]:
from collections import defaultdict

def calculate_relative_frequency_per_word_df(df, categories):
    word_frequency = defaultdict(lambda: defaultdict(float))
    total_per_year = df.groupby('year')['frequency'].sum().to_dict()
    
    for _, row in df.iterrows():
        year = int(row['year'])
        word = row['word'].lower()
        freq = row['frequency']
        total = total_per_year[year]
        
        for category, category_words in categories.items():
            if word in category_words:
                word_frequency[word][year] += freq / total
    
    return word_frequency


def relative_frequency_per_word_df(word_frequency):
    for category, words in categories.items():
        print(f"--{category}--")
        for word in words:
            if word in word_frequency:
                print(f"Relative frequency for the word '{word}':")
                for year, frequency in sorted(word_frequency[word].items()):
                    print(f"    Year {year}: {frequency:.2%}")
                print()


categories = {  
    'stench/stinking': ['stink', 'stinch', 'stench', 'reek', 'whiff', 'fetor', 'foetor', 'redolence', 'pong', 'niff', 'pungency', 'stinking', 'malodorous', 'fetid', 'foetid', 'niffy', 'smelly', 'reeking', 'whiffy', 'pungent', 'noisome', 'funky', 'musty', 'frowzy'],
    'fragrance/fragrant': ['redolence', 'perfume', 'scent', 'aroma', 'fragrance', 'musk', 'scented', 'aromatic', 'fragrant', 'redolent', 'sweet', 'fragrancy', 'odoriferousness'],
    'lacking_odour': ['odourless', 'odorless', 'scentless', 'unscented', 'deodorized', 'deodorization', 'deodorizer', 'deodorant', 'unsmelling', 'savourless', 'inodorate']
}


relative_word_frequency_per_year = calculate_relative_frequency_per_word_df(
    df_word_freq_year,
    categories
)

#relative_frequency_per_word_df(relative_word_frequency_per_year)

In [52]:
def print_top_words_per_category_per_year_raw(data, categories, top_n=10):
    """
    data: dict -> word -> year -> absolute frequency (raw count)
    categories: dict -> category -> list of words
    """
    for category, category_words in categories.items():
        years = set()
        for word in category_words:
            if word in data:
                years.update(data[word].keys())
        years = sorted(years)

        for year in years:
            category_words_year = {
                word: data[word][year]
                for word in category_words
                if word in data and year in data[word]
            }
            
            if not category_words_year:
                continue
            
            sorted_words = sorted(
                category_words_year.items(),
                key=lambda x: x[1],
                reverse=True
            )[:top_n]
            
            print(
                f"\nTop {top_n} words in the category "
                f"'{category}' for the year {year}:"
            )
            
            for word, freq in sorted_words:
                print(f"    {word}: {freq:.2%}")


# print_top_words_per_category_per_year_raw(
#     relative_word_frequency_per_year,
#     categories,
#     top_n=10
# )

In [67]:
import numpy as np
from scipy.stats import spearmanr

# Select the category to analyze
category = 'fragrance/fragrant'
category_words = categories[category]


years = sorted({
    year
    for word in category_words
    if word in relative_word_frequency_per_year
    for year in relative_word_frequency_per_year[word]
})

entropy_per_year = []
dominance_per_year = []

for year in years:

    freqs = [
        relative_word_frequency_per_year[word][year]
        for word in category_words
        if word in relative_word_frequency_per_year
        and year in relative_word_frequency_per_year[word]
    ]
    
    freqs = np.array(freqs)
    freqs = freqs / freqs.sum()  # Normalize for safety
    

    H = -np.sum(freqs * np.log2(freqs + 1e-10))
    entropy_per_year.append(H)

    dominance_per_year.append(freqs.max())


for year, H, dom in zip(years, entropy_per_year, dominance_per_year):
    print(f"Year {year}: Entropy={H:.4f}, Dominance={dom:.4f}")

rho, p = spearmanr(years, entropy_per_year)
print(f"\nSpearman rho for entropy over time: {rho:.3f}, p = {p:.3f}")


rho_d, p_d = spearmanr(years, dominance_per_year)
print(f"Spearman rho for dominance over time: {rho_d:.3f}, p = {p_d:.3f}")

Year 1: Entropy=1.7113, Dominance=0.5946
Year 2: Entropy=1.8729, Dominance=0.6000
Year 3: Entropy=1.9356, Dominance=0.5455
Year 4: Entropy=1.3509, Dominance=0.7481
Year 5: Entropy=2.1019, Dominance=0.4861
Year 6: Entropy=1.7807, Dominance=0.5903
Year 7: Entropy=1.9072, Dominance=0.5463
Year 8: Entropy=1.3758, Dominance=0.6020
Year 9: Entropy=2.1578, Dominance=0.4072
Year 10: Entropy=1.7878, Dominance=0.5423
Year 11: Entropy=1.8175, Dominance=0.4759
Year 12: Entropy=1.8265, Dominance=0.5486
Year 13: Entropy=1.9233, Dominance=0.5776
Year 14: Entropy=2.5412, Dominance=0.2621
Year 15: Entropy=2.6229, Dominance=0.2488
Year 16: Entropy=1.3451, Dominance=0.7200
Year 17: Entropy=1.7456, Dominance=0.5917
Year 18: Entropy=1.2934, Dominance=0.7293
Year 19: Entropy=1.8082, Dominance=0.4943
Year 20: Entropy=1.9244, Dominance=0.5941
Year 21: Entropy=1.9543, Dominance=0.5190
Year 22: Entropy=1.5983, Dominance=0.6806
Year 23: Entropy=1.6933, Dominance=0.6319
Year 24: Entropy=2.0544, Dominance=0.4812
Y

In [70]:
import numpy as np
from scipy.stats import spearmanr

# Select the category to analyze
category = 'stench/stinking'
category_words = categories[category]


years = sorted({
    year
    for word in category_words
    if word in relative_word_frequency_per_year
    for year in relative_word_frequency_per_year[word]
})

entropy_per_year = []
dominance_per_year = []

for year in years:

    freqs = [
        relative_word_frequency_per_year[word][year]
        for word in category_words
        if word in relative_word_frequency_per_year
        and year in relative_word_frequency_per_year[word]
    ]
    
    freqs = np.array(freqs)
    freqs = freqs / freqs.sum()  # Normalize for safety
    

    H = -np.sum(freqs * np.log2(freqs + 1e-10))
    entropy_per_year.append(H)

    dominance_per_year.append(freqs.max())


for year, H, dom in zip(years, entropy_per_year, dominance_per_year):
    print(f"Year {year}: Entropy={H:.4f}, Dominance={dom:.4f}")

rho, p = spearmanr(years, entropy_per_year)
print(f"\nSpearman rho for entropy over time: {rho:.3f}, p = {p:.3f}")


rho_d, p_d = spearmanr(years, dominance_per_year)
print(f"Spearman rho for dominance over time: {rho_d:.3f}, p = {p_d:.3f}")

Year 1: Entropy=-0.0000, Dominance=1.0000
Year 2: Entropy=-0.0000, Dominance=1.0000
Year 3: Entropy=0.9183, Dominance=0.6667
Year 4: Entropy=0.9183, Dominance=0.6667
Year 5: Entropy=-0.0000, Dominance=1.0000
Year 6: Entropy=-0.0000, Dominance=1.0000
Year 7: Entropy=0.9183, Dominance=0.6667
Year 9: Entropy=0.9183, Dominance=0.6667
Year 10: Entropy=-0.0000, Dominance=1.0000
Year 11: Entropy=-0.0000, Dominance=1.0000
Year 12: Entropy=1.2516, Dominance=0.6667
Year 13: Entropy=0.4690, Dominance=0.9000
Year 14: Entropy=1.0000, Dominance=0.5000
Year 15: Entropy=2.5411, Dominance=0.3600
Year 18: Entropy=0.8113, Dominance=0.7500
Year 19: Entropy=-0.0000, Dominance=1.0000
Year 20: Entropy=1.9219, Dominance=0.4000
Year 22: Entropy=-0.0000, Dominance=1.0000
Year 23: Entropy=-0.0000, Dominance=1.0000
Year 24: Entropy=1.5000, Dominance=0.5000
Year 25: Entropy=-0.0000, Dominance=1.0000
Year 26: Entropy=0.9183, Dominance=0.6667
Year 28: Entropy=1.3093, Dominance=0.6364
Year 29: Entropy=0.7219, Dominan

In [71]:
import numpy as np
from scipy.stats import spearmanr

# Select the category to analyze
category = 'lacking_odour'
category_words = categories[category]


years = sorted({
    year
    for word in category_words
    if word in relative_word_frequency_per_year
    for year in relative_word_frequency_per_year[word]
})

entropy_per_year = []
dominance_per_year = []

for year in years:

    freqs = [
        relative_word_frequency_per_year[word][year]
        for word in category_words
        if word in relative_word_frequency_per_year
        and year in relative_word_frequency_per_year[word]
    ]
    
    freqs = np.array(freqs)
    freqs = freqs / freqs.sum()  # Normalize for safety
    

    H = -np.sum(freqs * np.log2(freqs + 1e-10))
    entropy_per_year.append(H)

    dominance_per_year.append(freqs.max())


for year, H, dom in zip(years, entropy_per_year, dominance_per_year):
    print(f"Year {year}: Entropy={H:.4f}, Dominance={dom:.4f}")

rho, p = spearmanr(years, entropy_per_year)
print(f"\nSpearman rho for entropy over time: {rho:.3f}, p = {p:.3f}")


rho_d, p_d = spearmanr(years, dominance_per_year)
print(f"Spearman rho for dominance over time: {rho_d:.3f}, p = {p_d:.3f}")

Year 2: Entropy=0.7219, Dominance=0.8000
Year 3: Entropy=-0.0000, Dominance=1.0000
Year 5: Entropy=0.9710, Dominance=0.6000
Year 6: Entropy=0.9183, Dominance=0.6667
Year 7: Entropy=-0.0000, Dominance=1.0000
Year 8: Entropy=-0.0000, Dominance=1.0000
Year 9: Entropy=0.9544, Dominance=0.6250
Year 10: Entropy=0.9183, Dominance=0.6667
Year 11: Entropy=-0.0000, Dominance=1.0000
Year 12: Entropy=-0.0000, Dominance=1.0000
Year 13: Entropy=0.9852, Dominance=0.5714
Year 14: Entropy=-0.0000, Dominance=1.0000
Year 15: Entropy=1.3516, Dominance=0.5556
Year 16: Entropy=-0.0000, Dominance=1.0000
Year 17: Entropy=1.0000, Dominance=0.5000
Year 19: Entropy=-0.0000, Dominance=1.0000
Year 20: Entropy=0.9544, Dominance=0.6250
Year 21: Entropy=-0.0000, Dominance=1.0000
Year 22: Entropy=0.7219, Dominance=0.8000
Year 23: Entropy=0.9183, Dominance=0.6667
Year 24: Entropy=-0.0000, Dominance=1.0000
Year 25: Entropy=-0.0000, Dominance=1.0000
Year 26: Entropy=-0.0000, Dominance=1.0000
Year 27: Entropy=1.0000, Domi